# 3DP 비커스 경도 예측 모델 (Final)

**목표**: Ti-6Al-4V LPBF 공정 파라미터(스캔 속도, 층 두께, 레이저 파워, 에너지 밀도)로 비커스 경도(HV)를 예측한다.

**비교 실험**:
- 모델 A: 공정 파라미터 4개
- 모델 B: 공정 파라미터 + 기공률 (`OM_단면` 시트)
- 모델 C: 공정 파라미터 + 기공률 + OM 단면 이미지 특징
- 모델 D: 공정 파라미터 + 기공률 + 이미지 전처리/증강(crop, rotate) 특징 비교

**주의**: 데이터 수가 적을 가능성이 높으므로 LOOCV / 5-Fold CV를 중심으로 해석한다.

## 1. 라이브러리 import

필요한 라이브러리를 모두 불러오고, 한글 폰트(Malgun Gothic) 및 경고 메시지 처리를 설정한다.

In [ ]:
import os
import re
import glob
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# 한글 폰트 설정 (Windows)
mpl.rcParams['font.family'] = 'Malgun Gothic'
mpl.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')

# scikit-learn
from sklearn.model_selection import train_test_split, KFold, LeaveOneOut, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

# 이미지 처리
try:
    from PIL import Image
    HAS_PIL = True
except ImportError:
    HAS_PIL = False
    print('PIL을 사용할 수 없습니다. 이미지 특징 추출이 제한됩니다.')

print('라이브러리 import 완료')

## 2. 엑셀 파일 불러오기 (경도 시트)

지정된 파일이 없을 경우 같은 폴더의 다른 버전을 fallback으로 시도한다.
엑셀 헤더가 비정형(헤더 1행 + 단위 1행)이므로 첫 행을 헤더로, 두 번째 행은 단위로 처리한다.

In [ ]:
BASE_DIR = Path(r'C:\Users\user\Desktop\Projects\3dp')

candidate_files = [
    BASE_DIR / '데이터베이스_소재DB(4).xlsx',
    BASE_DIR / '데이터베이스_소재DB.xlsx',
]

EXCEL_PATH = None
for p in candidate_files:
    if p.exists():
        EXCEL_PATH = p
        break

print(f'사용할 엑셀 파일: {EXCEL_PATH}')
assert EXCEL_PATH is not None, '엑셀 파일을 찾을 수 없습니다.'

# 시트 목록 확인
xl = pd.ExcelFile(EXCEL_PATH)
print('시트 목록:', xl.sheet_names)

In [ ]:
def load_sheet_robust(path, sheet_name):
    """엑셀 시트를 비정형 헤더에 강건하게 로드.
    여러 header row 후보를 시도하고 가장 적합한 것을 사용."""
    best_df = None
    best_score = -1
    best_header = 0
    keywords = ['스캔', '층', '레이저', '파워', '에너지', '경도', '비커스', 'HV', '기공', 'porosity', 'Power', 'Energy']
    for h in range(0, 15):
        try:
            df_try = pd.read_excel(path, sheet_name=sheet_name, header=h)
        except Exception:
            continue
        cols = [str(c) for c in df_try.columns]
        score = sum(any(k in c for k in keywords) for c in cols)
        if score > best_score:
            best_score = score
            best_df = df_try
            best_header = h
    print(f"  -> 최적 header row = {best_header} (매칭 키워드 {best_score}개)")
    return best_df

def row_to_str_list(row):
    """Series의 값을 안전하게 문자열 리스트로 변환.
    pandas의 Series.astype(str)는 object dtype의 NaN을 'nan' 문자열로
    변환 못하는 경우가 있어, NaN을 명시적으로 빈 문자열로 처리."""
    return [('' if pd.isna(v) else str(v)) for v in row.tolist()]

df_hard_raw = load_sheet_robust(EXCEL_PATH, '경도')
# 컬럼명 strip
df_hard_raw.columns = [str(c).strip() for c in df_hard_raw.columns]

# 단위행(첫 데이터 행이 단위 정보일 가능성)이 데이터에 섞여있을 수 있으므로 제거
first_row = row_to_str_list(df_hard_raw.iloc[0])
if any(u in ' '.join(first_row) for u in ['µm', 'mm/s', 'J/mm^3', 'HV', 'HRA']):
    print('첫 행이 단위 정보로 판단되어 제거합니다.')
    df_hard_raw = df_hard_raw.iloc[1:].reset_index(drop=True)

print('shape:', df_hard_raw.shape)
print('컬럼명:', list(df_hard_raw.columns))
df_hard_raw.head()

## 3. 컬럼명 자동 매칭

컬럼명이 `스캔 속도`, `층 두께`, `레이저 파워`, `에너지 밀도`, `비커스 경도` 와 정확히 일치하지 않을 수 있으므로 키워드 기반으로 자동 매칭한다.

In [ ]:
def find_col(df, keywords, exclude=None):
    """keyword가 모두/하나라도 포함된 컬럼을 찾는다."""
    exclude = exclude or []
    cols = list(df.columns)
    for kw_set in keywords:
        # kw_set: tuple of must-have substrings
        for c in cols:
            cs = str(c)
            if any(ex in cs for ex in exclude):
                continue
            if all(k.lower() in cs.lower() for k in kw_set):
                return c
    return None

col_scan  = find_col(df_hard_raw, [('스캔', '속도'), ('scan', 'speed'), ('속도',)])
col_layer = find_col(df_hard_raw, [('층', '두께'), ('layer', 'thick'), ('두께',)])
col_power = find_col(df_hard_raw, [('레이저', '파워'), ('laser', 'power'), ('파워',), ('power',)])
col_ed    = find_col(df_hard_raw, [('에너지', '밀도'), ('energy', 'density'), ('에너지',)])
col_hv    = find_col(df_hard_raw, [('비커스', '경도'), ('비커스',), ('vickers',), ('hv',), ('경도',)],
                    exclude=['로크웰', 'HRA'])

print('--- 자동 매칭 결과 ---')
print(f'스캔 속도   : {col_scan}')
print(f'층 두께     : {col_layer}')
print(f'레이저 파워 : {col_power}')
print(f'에너지 밀도 : {col_ed}')
print(f'비커스 경도 : {col_hv}')

INPUT_COLS = [col_scan, col_layer, col_power, col_ed]
TARGET_COL = col_hv

missing = [n for n, c in zip(['scan','layer','power','ed','hv'], INPUT_COLS+[TARGET_COL]) if c is None]
if missing:
    print(f'⚠ 매칭 실패: {missing} → 노트북 진행 중에도 확인 필요')

## 4. 데이터 전처리

- 입력 변수와 타겟만 추출
- 숫자형 변환 후 결측치 제거
- IQR 기반 이상치 확인 (제거는 보수적으로 하지 않고 출력만)
- 샘플 식별자 컬럼이 있으면 함께 보관 (이미지/기공률 매칭용)

In [ ]:
# 샘플 식별자 컬럼 자동 탐색 (TIT2P1SS1 같은 코드)
def find_id_col(df):
    for c in df.columns:
        sample_vals = df[c].dropna().astype(str).head(20).tolist()
        if any(re.match(r'^TIT\d+P\d+SS\d+', v) for v in sample_vals):
            return c
    return None

id_col = find_id_col(df_hard_raw)
print('샘플 ID 컬럼:', id_col)

use_cols = INPUT_COLS + [TARGET_COL]
if id_col is not None:
    use_cols = [id_col] + use_cols

df_hard = df_hard_raw[use_cols].copy()

# 숫자 변환
for c in INPUT_COLS + [TARGET_COL]:
    df_hard[c] = pd.to_numeric(df_hard[c], errors='coerce')

before = len(df_hard)
df_hard = df_hard.dropna(subset=INPUT_COLS + [TARGET_COL]).reset_index(drop=True)
after = len(df_hard)
print(f'결측치 제거: {before} → {after} 행')

# 이상치 간단 확인 (IQR)
print('\n--- 기술통계 ---')
print(df_hard[INPUT_COLS + [TARGET_COL]].describe().round(2))

for c in INPUT_COLS + [TARGET_COL]:
    q1, q3 = df_hard[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((df_hard[c] < lo) | (df_hard[c] > hi)).sum()
    print(f'{c}: 이상치 후보 {n_out}개 (lo={lo:.2f}, hi={hi:.2f})')

df_hard.head()

## 5. 탐색적 분석 (EDA)

입력 변수 및 타겟의 분포, 산점도, 상관관계 heatmap을 시각화한다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.ravel(), INPUT_COLS + [TARGET_COL]):
    ax.hist(df_hard[col], bins=15, edgecolor='black', alpha=0.7)
    ax.set_title(f'분포: {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('빈도')
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, INPUT_COLS):
    ax.scatter(df_hard[col], df_hard[TARGET_COL], alpha=0.6)
    ax.set_xlabel(col)
    ax.set_ylabel(TARGET_COL)
    ax.set_title(f'{col} vs {TARGET_COL}')
plt.tight_layout()
plt.show()

In [ ]:
corr = df_hard[INPUT_COLS + [TARGET_COL]].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', color='black')
plt.colorbar(im, ax=ax)
plt.title('상관관계 Heatmap')
plt.tight_layout()
plt.show()

## 6. 모델 정의 및 평가 함수

비교할 모델을 Pipeline으로 묶고 평가 함수(R², MAE, RMSE)를 정의한다.
스케일링이 필요한 모델(Linear/Ridge/Lasso/SVR/GPR)은 `StandardScaler` + 모델 Pipeline으로 구성한다.

In [ ]:
def build_models(random_state=42):
    models = {
        'LinearRegression': Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
        'Ridge'           : Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))]),
        'Lasso'           : Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=0.1, max_iter=10000))]),
        'RandomForest'    : RandomForestRegressor(n_estimators=300, random_state=random_state),
        'SVR'             : Pipeline([('sc', StandardScaler()), ('m', SVR(C=10, gamma='scale'))]),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=200, random_state=random_state),
    }
    # GPR은 데이터가 매우 적으면 잘 동작하지 않을 수 있으므로 try/except로 추가
    try:
        kernel = C(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0)
        models['GaussianProcess'] = Pipeline([
            ('sc', StandardScaler()),
            ('m', GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=random_state))
        ])
    except Exception as e:
        print('GPR 추가 실패:', e)
    return models

def eval_split(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        'R2'  : r2_score(y_test, pred),
        'MAE' : mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
    }

def eval_cv(model, X, y, cv):
    """cross_val_predict로 R2/MAE/RMSE 한번에 계산."""
    pred = cross_val_predict(model, X, y, cv=cv)
    return {
        'R2'  : r2_score(y, pred),
        'MAE' : mean_absolute_error(y, pred),
        'RMSE': np.sqrt(mean_squared_error(y, pred)),
    }

def evaluate_all(X, y, label='', test_size=0.2, n_splits=5, do_loocv=True, random_state=42):
    """모든 모델을 Train/Test, KFold, LOOCV로 평가하여 표(DataFrame)로 반환."""
    rows = []
    n = len(X)
    can_loocv = do_loocv and n <= 200
    if n < 10:
        print(f'  ⚠ 데이터 수 {n}개 — Train/Test split이 불안정할 수 있음')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    kf = KFold(n_splits=min(n_splits, n), shuffle=True, random_state=random_state)
    loo = LeaveOneOut() if can_loocv else None
    for name, model in build_models(random_state).items():
        try:
            sp  = eval_split(model, X_train, X_test, y_train, y_test)
            kfm = eval_cv(model, X, y, kf)
            loocv = eval_cv(model, X, y, loo) if loo is not None else {'R2': np.nan, 'MAE': np.nan, 'RMSE': np.nan}
            rows.append({
                'feature_set': label, 'model': name,
                'split_R2': sp['R2'], 'split_MAE': sp['MAE'], 'split_RMSE': sp['RMSE'],
                'kfold_R2': kfm['R2'], 'kfold_MAE': kfm['MAE'], 'kfold_RMSE': kfm['RMSE'],
                'loocv_R2': loocv['R2'], 'loocv_MAE': loocv['MAE'], 'loocv_RMSE': loocv['RMSE'],
            })
        except Exception as e:
            print(f'  ⚠ {name} 평가 실패: {e}')
    return pd.DataFrame(rows)

print('모델/평가 함수 준비 완료')

## 7. 모델 A 평가 — 공정 파라미터만

`스캔 속도, 층 두께, 레이저 파워, 에너지 밀도` 4개 입력으로 비커스 경도 예측.

In [ ]:
X_A = df_hard[INPUT_COLS].values
y_A = df_hard[TARGET_COL].values
print(f'모델 A 입력 shape: {X_A.shape}, 타겟 shape: {y_A.shape}')

results_A = evaluate_all(X_A, y_A, label='A_processOnly')
print('\n--- 모델 A 결과 ---')
print(results_A.round(4).to_string(index=False))

## 7-1. 다중공선성 진단 — Model A' / A'' / A 비교

`ED = P / (v · h · t)` 이므로 입력 4개의 실효 자유도가 3 미만일 가능성이 큼.
같은 모델(SVR/RF/Ridge)을 다음 4개 feature set으로 5-Fold CV 비교:

- **A_ED**       : 에너지 밀도 1개만
- **A_PvH**      : (레이저 파워, 스캔 속도, 층 두께) 3개만
- **A_full**     : 4개 모두 (현재 baseline)
- **A_derived**  : (ED, log_ED, P/v) — 비선형 파생 변수만

VIF도 같이 계산해 다중공선성을 수치로 확인. ED-only가 A_full과 비슷하거나 더 나으면 → "4개 입력 중 3개는 노이즈"라는 결론이 확정된다.

In [ ]:
# =============================================================
# 7-1. 다중공선성 진단: ED-only / PvH-only / Full / Derived
# =============================================================

# (1) VIF 계산 (Variance Inflation Factor)
#     VIF = 1 / (1 - R²(X_i ~ 나머지))
#     >10 = 심각, 5~10 = 경고, <5 = OK
print('=== VIF (다중공선성 지표) ===')
X_full_df = df_hard[INPUT_COLS].copy()
for i, col in enumerate(INPUT_COLS):
    others = [c for c in INPUT_COLS if c != col]
    yi = X_full_df[col].values
    Xo = X_full_df[others].values
    # 선형 회귀로 R² 계산
    pipe_vif = Pipeline([('sc', StandardScaler()), ('m', LinearRegression())])
    pipe_vif.fit(Xo, yi)
    r2_i = r2_score(yi, pipe_vif.predict(Xo))
    vif = 1.0 / (1.0 - r2_i) if r2_i < 0.9999 else float('inf')
    flag = '🔴' if vif > 10 else ('🟠' if vif > 5 else '🟢')
    print(f'  {flag} {col:25s}: VIF = {vif:8.2f}  (R² ~ 나머지 = {r2_i:.4f})')

# (2) ED ≈ P/(v·h) 관계식 검증 (Hatch spacing t는 상수 가정)
print('\n=== ED 정의식 일치도: ED vs P/(scan·layer) ===')
ed_actual = df_hard[col_ed].values
ratio = df_hard[col_power].values / (df_hard[col_scan].values * df_hard[col_layer].values * 1e-3)
corr_ed_def = np.corrcoef(ed_actual, ratio)[0, 1]
print(f'  corr(ED, P/(v·layer)) = {corr_ed_def:.4f}  (1에 가까우면 ED는 다른 3개의 함수)')

# (3) 4가지 feature set 비교
print('\n=== Feature set별 5-Fold CV 비교 ===')
y_proc = df_hard[TARGET_COL].values
P = df_hard[col_power].values
v = df_hard[col_scan].values
h = df_hard[col_layer].values
ED = df_hard[col_ed].values

feature_sets = {
    "A_ED       (1 var)": np.column_stack([ED]),
    "A_PvH      (3 var)": np.column_stack([P, v, h]),
    "A_full     (4 var)": np.column_stack([v, h, P, ED]),
    "A_derived  (3 var)": np.column_stack([ED, np.log(ED), P/v]),
}

models_to_test = {
    'Ridge'       : Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))]),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=42),
    'SVR'         : Pipeline([('sc', StandardScaler()), ('m', SVR(C=10, gamma='scale'))]),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for fs_name, X_fs in feature_sets.items():
    for m_name, mdl in models_to_test.items():
        pred = cross_val_predict(mdl, X_fs, y_proc, cv=kf)
        r2  = r2_score(y_proc, pred)
        mae = mean_absolute_error(y_proc, pred)
        rmse = np.sqrt(mean_squared_error(y_proc, pred))
        rows.append({'feature_set': fs_name, 'model': m_name,
                     'n_features': X_fs.shape[1],
                     'kfold_R2': r2, 'kfold_MAE': mae, 'kfold_RMSE': rmse})
df_diag = pd.DataFrame(rows)
print(df_diag.round(4).to_string(index=False))

# (4) 결론 자동 출력
print('\n=== 진단 결론 ===')
best_per_set = df_diag.groupby('feature_set')['kfold_R2'].max()
print('Feature set별 최고 R²:')
print(best_per_set.round(4).to_string())
delta_full_vs_ed = best_per_set.get('A_full     (4 var)', 0) - best_per_set.get('A_ED       (1 var)', 0)
print(f'\nΔR² (full − ED-only) = {delta_full_vs_ed:+.4f}')
if abs(delta_full_vs_ed) < 0.03:
    print('→ 4변수와 ED 1변수 성능 차이 미미. **다중공선성으로 추가 정보 없음**.')
    print('→ 결론: 4개 입력은 사실상 1~2개 자유도. 변수를 추가해도 R² 개선 한계.')
elif delta_full_vs_ed > 0.05:
    print('→ 4변수가 의미 있게 더 나음. 다중공선성보다 비선형 정보가 우세.')
else:
    print('→ 모호. 다른 노이즈 요인이 더 큰 영향.')

## 8. OM_단면 시트 불러오기 (기공률 추가)

기공률 또는 porosity가 포함된 컬럼을 자동 탐색하고, 경도 시트와 병합 가능한 키(샘플 ID)를 찾는다.

In [ ]:
om_sheet_name = None
for s in xl.sheet_names:
    if 'OM' in s or '단면' in s or 'porosity' in s.lower() or '기공' in s:
        om_sheet_name = s
        break
print(f'OM 시트명: {om_sheet_name}')

df_om = None
if om_sheet_name is not None:
    df_om = load_sheet_robust(EXCEL_PATH, om_sheet_name)
    df_om.columns = [str(c).strip() for c in df_om.columns]
    # 단위행 제거 시도 (NaN-safe 변환 사용)
    first_row = row_to_str_list(df_om.iloc[0])
    if any(u in ' '.join(first_row) for u in ['µm', 'mm/s', 'J/mm^3', '%', 'HV']):
        df_om = df_om.iloc[1:].reset_index(drop=True)
    print('OM 시트 shape:', df_om.shape)
    print('OM 시트 컬럼:', list(df_om.columns))
    df_om.head()
else:
    print('⚠ OM_단면 시트를 찾지 못했습니다.')

In [ ]:
# 기공률 컬럼 찾기
col_porosity = None
om_id_col = None
if df_om is not None:
    col_porosity = find_col(df_om, [('기공률',), ('기공',), ('porosity',), ('void',)])
    om_id_col    = find_id_col(df_om)
    print(f'기공률 컬럼 : {col_porosity}')
    print(f'OM ID 컬럼  : {om_id_col}')

In [ ]:
# 경도 시트와 병합: 샘플 ID 우선, 안되면 (Power, Scan speed, Layer thickness)로 join
df_AB = None
if df_om is not None and col_porosity is not None:
    df_om2 = df_om.copy()
    df_om2[col_porosity] = pd.to_numeric(df_om2[col_porosity], errors='coerce')

    if id_col is not None and om_id_col is not None:
        # 같은 샘플 ID가 여러 줄이면 평균
        om_agg = df_om2.groupby(om_id_col, as_index=False)[col_porosity].mean()
        df_AB = df_hard.merge(om_agg, left_on=id_col, right_on=om_id_col, how='left')
        if id_col != om_id_col and om_id_col in df_AB.columns:
            df_AB = df_AB.drop(columns=[om_id_col])
        print(f'샘플 ID 기반 병합: {df_AB[col_porosity].notna().sum()}/{len(df_AB)} 행 매칭')
    else:
        # 공정 파라미터로 병합 시도
        common = [c for c in INPUT_COLS if c in df_om2.columns]
        if len(common) >= 2:
            print(f'공정 파라미터 {common}로 병합 시도')
            df_AB = df_hard.merge(df_om2[common + [col_porosity]], on=common, how='left')
        else:
            print('⚠ 공통 키를 찾지 못해 병합 불가. OM 시트 컬럼 직접 확인 필요')
            print('경도 컬럼:', list(df_hard.columns))
            print('OM 컬럼  :', list(df_om2.columns))

if df_AB is not None:
    print('\n병합 후 shape:', df_AB.shape)
    print(f'기공률 결측: {df_AB[col_porosity].isna().sum()}개')
    df_AB.head()

## 9. 모델 B 평가 — 공정 파라미터 + 기공률

In [ ]:
results_B = pd.DataFrame()
df_B = None
if df_AB is not None and col_porosity is not None:
    df_B = df_AB.dropna(subset=INPUT_COLS + [col_porosity, TARGET_COL]).reset_index(drop=True)
    print(f'모델 B 사용 데이터: {len(df_B)}행')
    if len(df_B) >= 5:
        X_B = df_B[INPUT_COLS + [col_porosity]].values
        y_B = df_B[TARGET_COL].values
        results_B = evaluate_all(X_B, y_B, label='B_process+porosity')
        print('\n--- 모델 B 결과 ---')
        print(results_B.round(4).to_string(index=False))
    else:
        print('⚠ 데이터 부족 — 모델 B 스킵')
else:
    print('⚠ 기공률 데이터를 사용할 수 없어 모델 B 스킵')

## 10. OM 단면 이미지 불러오기 및 매칭

`이미지_OM_단면` 폴더의 이미지를 샘플 ID와 매칭한다.
파일명 패턴: `TIT2P1SS1_1.jpg` → 샘플 ID = `TIT2P1SS1`, 반복 번호 = 1

In [ ]:
IMG_DIR = BASE_DIR / '이미지_OM_단면'
print('이미지 폴더 존재:', IMG_DIR.exists())

image_paths = []
for ext in ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff', '*.JPG', '*.JPEG', '*.PNG']:
    image_paths.extend(IMG_DIR.glob(ext))
image_paths = [p for p in image_paths if 'thumbs' not in p.name.lower()]
print(f'이미지 파일 수: {len(image_paths)}')

def parse_sample_id(filename):
    """파일명에서 샘플 ID 추출. e.g. 'TIT2P1SS1_1.jpg' -> 'TIT2P1SS1'"""
    name = Path(filename).stem
    m = re.match(r'(TIT\d+P\d+SS\d+)', name)
    if m:
        return m.group(1)
    return name

img_records = [{'path': str(p), 'filename': p.name, 'sample_id': parse_sample_id(p.name)} for p in image_paths]
df_img = pd.DataFrame(img_records)
print('샘플 ID 종류:', df_img['sample_id'].nunique())
df_img.head()

In [ ]:
# 매칭 상태 확인
if id_col is not None:
    hard_ids = set(df_hard[id_col].astype(str).unique())
    img_ids  = set(df_img['sample_id'].unique())
    matched = hard_ids & img_ids
    only_hard = hard_ids - img_ids
    only_img  = img_ids - hard_ids
    print(f'매칭된 샘플 : {len(matched)}')
    print(f'경도만 있음 : {len(only_hard)} → {sorted(list(only_hard))[:10]} ...')
    print(f'이미지만 있음: {len(only_img)} → {sorted(list(only_img))[:10]} ...')
else:
    print('⚠ 경도 데이터에 샘플 ID 컬럼이 없어 이미지 매칭 불가')

## 11. 이미지 특징 추출 함수 (handcrafted)

데이터 수가 적을 가능성이 높아 CNN 대신 단순 통계 기반 특징을 추출한다.
- grayscale 평균, 표준편차
- 어두운/밝은 픽셀 비율 (porosity 추정에 도움)
- edge density (Sobel 근사)
- texture contrast (이웃 픽셀 차이)
- 이미지 크기, aspect ratio

In [ ]:
def extract_features(img_arr):
    """2D ndarray (grayscale, 0-255) → feature dict"""
    arr = img_arr.astype(np.float32)
    H, W = arr.shape[:2]
    feat = {}
    feat['mean']      = float(arr.mean())
    feat['std']       = float(arr.std())
    feat['dark_frac'] = float((arr < 60).mean())
    feat['bright_frac'] = float((arr > 200).mean())
    # Sobel 근사 edge
    gx = np.zeros_like(arr); gy = np.zeros_like(arr)
    gx[:, 1:-1] = arr[:, 2:] - arr[:, :-2]
    gy[1:-1, :] = arr[2:, :] - arr[:-2, :]
    grad = np.sqrt(gx*gx + gy*gy)
    feat['edge_density'] = float((grad > 30).mean())
    feat['edge_mean']    = float(grad.mean())
    # texture contrast (수평 이웃 차이의 std)
    feat['texture_contrast'] = float(np.abs(arr[:, 1:] - arr[:, :-1]).mean())
    feat['height'] = H
    feat['width']  = W
    feat['aspect'] = W / H if H > 0 else 0
    return feat

def load_and_preprocess(path, mode='original', target_size=256):
    """이미지를 PIL로 로드하고 mode에 따라 전처리."""
    img = Image.open(path).convert('L')
    if mode == 'original':
        pass
    elif mode == 'resize':
        img = img.resize((target_size, target_size))
    elif mode == 'center_crop':
        w, h = img.size
        s = min(w, h)
        l = (w - s) // 2; t = (h - s) // 2
        img = img.crop((l, t, l+s, t+s)).resize((target_size, target_size))
    elif mode == 'rot90':
        img = img.rotate(90, expand=True).resize((target_size, target_size))
    elif mode == 'rot180':
        img = img.rotate(180, expand=True).resize((target_size, target_size))
    elif mode == 'rot270':
        img = img.rotate(270, expand=True).resize((target_size, target_size))
    return np.array(img)

def extract_dataset_features(df_img, mode='original'):
    """이미지 DF에 대해 특징 추출 후 sample_id별로 평균(반복 측정 통합)."""
    rows = []
    for _, r in df_img.iterrows():
        try:
            arr = load_and_preprocess(r['path'], mode=mode)
            f = extract_features(arr)
            f['sample_id'] = r['sample_id']
            rows.append(f)
        except Exception as e:
            pass
    df_feat = pd.DataFrame(rows)
    if len(df_feat) == 0:
        return df_feat
    return df_feat.groupby('sample_id', as_index=False).mean(numeric_only=True)

print('이미지 특징 추출 함수 준비 완료')

## 12. 모델 C 평가 — 공정 파라미터 + 기공률 + 이미지 특징(원본)

In [ ]:
results_C = pd.DataFrame()
df_C = None
img_feat_cols = []
if HAS_PIL and id_col is not None and df_AB is not None and col_porosity is not None:
    print('이미지 특징 추출 중 (mode=original) ...')
    df_img_feat = extract_dataset_features(df_img, mode='original')
    print(f'  추출 완료: {df_img_feat.shape}')
    img_feat_cols = [c for c in df_img_feat.columns if c != 'sample_id']

    df_C = df_AB.merge(df_img_feat, left_on=id_col, right_on='sample_id', how='left')
    if id_col != 'sample_id' and 'sample_id' in df_C.columns:
        df_C = df_C.drop(columns=['sample_id'])
    df_C = df_C.dropna(subset=INPUT_COLS + [col_porosity, TARGET_COL] + img_feat_cols).reset_index(drop=True)
    print(f'모델 C 사용 데이터: {len(df_C)}행')

    if len(df_C) >= 5:
        feat_cols_C = INPUT_COLS + [col_porosity] + img_feat_cols
        X_C = df_C[feat_cols_C].values
        y_C = df_C[TARGET_COL].values
        results_C = evaluate_all(X_C, y_C, label='C_process+porosity+imgOriginal')
        print('\n--- 모델 C 결과 ---')
        print(results_C.round(4).to_string(index=False))
    else:
        print('⚠ 데이터 부족 — 모델 C 스킵')
else:
    print('⚠ 이미지/기공률/ID 중 하나 이상 부족 — 모델 C 스킵')

## 13. 모델 D 평가 — 이미지 전처리/증강 방식 비교

**중요**: 회전/crop된 이미지를 “새 샘플”로 추가하면 train/test에 같은 샘플이 들어가 leakage가 발생한다.
따라서 여기서는 **특징 추출 방식 비교**만 수행한다 (원본/resize/center_crop/rot90/rot180/rot270).

In [ ]:
results_D = pd.DataFrame()
if HAS_PIL and id_col is not None and df_AB is not None and col_porosity is not None:
    modes = ['original', 'resize', 'center_crop', 'rot90', 'rot180', 'rot270']
    rows = []
    for mode in modes:
        print(f'\n>>> 이미지 mode = {mode}')
        df_feat = extract_dataset_features(df_img, mode=mode)
        cols_feat = [c for c in df_feat.columns if c != 'sample_id']
        df_m = df_AB.merge(df_feat, left_on=id_col, right_on='sample_id', how='left')
        if id_col != 'sample_id' and 'sample_id' in df_m.columns:
            df_m = df_m.drop(columns=['sample_id'])
        df_m = df_m.dropna(subset=INPUT_COLS + [col_porosity, TARGET_COL] + cols_feat).reset_index(drop=True)
        if len(df_m) < 5:
            print('  데이터 부족, 스킵')
            continue
        X_m = df_m[INPUT_COLS + [col_porosity] + cols_feat].values
        y_m = df_m[TARGET_COL].values
        # 모드별 비교는 핵심 모델 4개로 제한
        for name in ['Ridge', 'RandomForest', 'GradientBoosting', 'SVR']:
            mdl = build_models()[name]
            kf = KFold(n_splits=min(5, len(df_m)), shuffle=True, random_state=42)
            try:
                m = eval_cv(mdl, X_m, y_m, kf)
                rows.append({'image_mode': mode, 'model': name, 'n': len(df_m), **m})
            except Exception as e:
                print(f'  {name} 실패: {e}')
    results_D = pd.DataFrame(rows)
    print('\n--- 모델 D 결과 (5-Fold CV) ---')
    if len(results_D):
        print(results_D.round(4).to_string(index=False))
else:
    print('⚠ 모델 D 실행 불가')

## 14. 통합 결과 비교

모델 A, B, C 결과를 한 표로 합치고 5-fold CV 기준으로 시각화한다.

In [ ]:
all_results = pd.concat([results_A, results_B, results_C], ignore_index=True)
print('--- 전체 결과 (A, B, C) ---')
print(all_results.round(4).to_string(index=False))

In [ ]:
# 모델별 5-fold R^2 비교 (feature_set별 그룹)
if len(all_results):
    pivot = all_results.pivot_table(index='model', columns='feature_set', values='kfold_R2')
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot(kind='bar', ax=ax)
    ax.set_ylabel('5-Fold CV R²')
    ax.set_title('모델 × Feature set 별 R² 비교')
    ax.axhline(0, color='k', lw=0.5)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

# 모델 D: 이미지 mode 별 비교
if len(results_D):
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot_d = results_D.pivot_table(index='image_mode', columns='model', values='R2')
    pivot_d.plot(kind='bar', ax=ax)
    ax.set_ylabel('5-Fold CV R²')
    ax.set_title('이미지 전처리 mode 별 성능 (5-Fold CV)')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()

## 15. 최고 성능 모델 자동 선택 + 실제값/예측값/잔차 시각화

5-Fold CV의 R² 기준으로 최고 모델을 선택한다.

In [ ]:
best_row = all_results.sort_values('kfold_R2', ascending=False).iloc[0]
print('--- 최고 성능 ---')
# best_row는 문자열(feature_set, model)과 숫자(R2/MAE/RMSE)가 섞여있어
# Series.round()가 문자열에서 실패하므로 숫자 컬럼만 라운딩
numeric_cols = all_results.select_dtypes(include=['number']).columns
print(f'  feature_set : {best_row["feature_set"]}')
print(f'  model       : {best_row["model"]}')
print(best_row[numeric_cols].round(4))

best_label = best_row['feature_set']
best_name  = best_row['model']

# 해당 feature set의 X, y 재구성
if best_label == 'A_processOnly':
    X_best, y_best = X_A, y_A
    feat_names = INPUT_COLS
elif best_label == 'B_process+porosity':
    X_best = df_B[INPUT_COLS + [col_porosity]].values
    y_best = df_B[TARGET_COL].values
    feat_names = INPUT_COLS + [col_porosity]
else:  # C
    feat_names = INPUT_COLS + [col_porosity] + img_feat_cols
    X_best = df_C[feat_names].values
    y_best = df_C[TARGET_COL].values

best_model = build_models()[best_name]
y_pred_cv = cross_val_predict(best_model, X_best, y_best, cv=KFold(n_splits=5, shuffle=True, random_state=42))
best_model.fit(X_best, y_best)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_best, y_pred_cv, alpha=0.7)
lims = [min(y_best.min(), y_pred_cv.min()), max(y_best.max(), y_pred_cv.max())]
axes[0].plot(lims, lims, 'r--')
axes[0].set_xlabel('실제 비커스 경도')
axes[0].set_ylabel('예측 비커스 경도')
axes[0].set_title(f'실제 vs 예측 ({best_name}, {best_label})')

resid = y_best - y_pred_cv
axes[1].scatter(y_pred_cv, resid, alpha=0.7)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel('예측값')
axes[1].set_ylabel('잔차 (실제 - 예측)')
axes[1].set_title('잔차 plot')
plt.tight_layout()
plt.show()

## 16. Feature Importance / Permutation Importance

In [ ]:
# feature importance: tree 기반 모델에서 가장 잘 보임 — 별도로 RandomForest 학습
rf_imp_model = RandomForestRegressor(n_estimators=300, random_state=42)
rf_imp_model.fit(X_best, y_best)
imp = rf_imp_model.feature_importances_
imp_df = pd.DataFrame({'feature': feat_names, 'importance': imp}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, max(3, 0.3*len(feat_names))))
ax.barh(imp_df['feature'], imp_df['importance'])
ax.set_title('RandomForest Feature Importance')
ax.set_xlabel('importance')
plt.tight_layout()
plt.show()
print(imp_df.sort_values('importance', ascending=False).to_string(index=False))

In [ ]:
# permutation importance (best_model 기반)
try:
    perm = permutation_importance(best_model, X_best, y_best, n_repeats=20, random_state=42, n_jobs=-1)
    perm_df = pd.DataFrame({
        'feature': feat_names,
        'perm_importance_mean': perm.importances_mean,
        'perm_importance_std' : perm.importances_std,
    }).sort_values('perm_importance_mean', ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(3, 0.3*len(feat_names))))
    ax.barh(perm_df['feature'], perm_df['perm_importance_mean'],
            xerr=perm_df['perm_importance_std'])
    ax.set_title(f'Permutation Importance ({best_name})')
    plt.tight_layout()
    plt.show()
    print(perm_df.sort_values('perm_importance_mean', ascending=False).to_string(index=False))
except Exception as e:
    print('permutation importance 계산 실패:', e)

## 17. 결과 저장 (CSV + 최고 모델 pkl)

In [ ]:
results_path = BASE_DIR / '3dp_new_final_model_results.csv'
all_results.to_csv(results_path, index=False, encoding='utf-8-sig')
print('결과표 저장:', results_path)

if len(results_D):
    results_d_path = BASE_DIR / '3dp_new_final_image_mode_results.csv'
    results_D.to_csv(results_d_path, index=False, encoding='utf-8-sig')
    print('이미지 mode 결과 저장:', results_d_path)

model_path = BASE_DIR / '3dp_new_final_best_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({
        'model': best_model,
        'feature_names': feat_names,
        'feature_set': best_label,
        'model_name': best_name,
        'metrics': best_row.to_dict(),
    }, f)
print('최고 모델 저장:', model_path)

## 18. 해석 및 최종 요약

### 변수 기여도 해석
- 일반적으로 LPBF Ti-6Al-4V에서는 **에너지 밀도**가 미세조직(α'/α+β 비율)과 결함 형성을 동시에 통제하므로 경도와 가장 밀접하다.
- **레이저 파워**, **스캔 속도**는 에너지 밀도에 종속이므로 단독 기여도는 낮을 수 있다.
- **층 두께**는 열 누적과 결함률에 영향을 줘 보조적인 변수로 작용한다.
- **기공률**을 추가하면 같은 에너지 밀도에서도 결함의 영향을 분리할 수 있어 모델 B/C에서 R²가 향상되는지 확인.
- **이미지 특징**은 기공/조직의 시각적 정보를 반영하지만, handcrafted feature는 OM 단면의 조명/배율 차이에 민감해 항상 도움되지는 않는다.

### R²가 낮을 경우 가능한 원인
1. **데이터 수 부족**: 샘플당 몇십 개 수준이면 LOOCV/5-Fold 모두 분산이 매우 크다.
2. **공정 파라미터 4개만으로는 부족**: 미세조직(상분포, 결정립 크기), 잔류응력 등이 빠져 있어 같은 입력에 대해 경도가 흩어질 수 있다.
3. **경도 측정 오차**: HV 측정 위치가 기공/조직 경계와 겹치면 분산 증가.
4. **비선형성 / 상호작용**: 에너지 밀도와 층 두께의 비선형 결합을 선형 모델은 포착하지 못함 → RF/GBR/GPR 우위.
5. **에너지 밀도 다중공선성**: ED = P/(v·h·t) 이므로 다른 3변수에 종속 → Ridge/Lasso로 완화 필요.
6. **이미지 매칭 누락**: 일부 샘플의 OM 이미지 결측 → 평균 기공률 정보 손실.

### 이미지 특징을 추가했을 때 성능 변화
- 향상되었다면: 기공/조직의 시각 정보가 공정 파라미터로는 표현되지 않는 분산을 설명한 것.
- 향상되지 않았다면: (1) handcrafted feature가 도메인 정보(예: pore size 분포)를 충분히 못 잡거나, (2) 이미지 수가 적어 추가 특징이 노이즈로 작용했을 가능성. 이 경우 CNN/segmentation 기반 pore 정량화로 확장하는 것이 적절하다.
- 회전/crop 모드 비교에서 큰 차이가 없다면 → 통계 기반 특징은 회전 불변성에 가깝다는 의미이며, 새 정보가 거의 없음을 시사.

In [ ]:
# 최종 요약 출력
top_feat = imp_df.sort_values('importance', ascending=False).head(3)['feature'].tolist()
summary = f"""
================ 최종 요약 ================
가장 좋은 모델     : {best_name}
사용 Feature set   : {best_label}
5-Fold R²          : {best_row['kfold_R2']:.4f}
5-Fold MAE         : {best_row['kfold_MAE']:.4f}
5-Fold RMSE        : {best_row['kfold_RMSE']:.4f}
LOOCV R²           : {best_row.get('loocv_R2', float('nan')):.4f}
Train/Test R²      : {best_row['split_R2']:.4f}
주요 변수 (RF top3): {', '.join(top_feat)}
"""
print(summary)

## 최종 결론 (Markdown)

| 항목 | 값 |
| --- | --- |
| 가장 좋은 모델 | 위 셀 출력의 `best_name` |
| 사용 Feature set | A / B / C 중 자동 선택 |
| 최종 R² (5-Fold CV) | 위 셀 출력 참조 |
| MAE | 위 셀 출력 참조 |
| RMSE | 위 셀 출력 참조 |
| 중요 변수 | RandomForest feature importance 상위 3 |

**비교 실험 요지**
- 모델 A → B: 기공률 추가 시 모델 성능 변화 확인
- 모델 B → C: 이미지 handcrafted 특징이 기공률 외 추가 정보를 주는지 확인
- 모델 D: 이미지 전처리(원본/resize/crop/회전)가 특징 추출에 미치는 영향 비교 — 회전/crop은 *데이터 증강이 아닌 특징 추출 방식 비교*로만 사용해 leakage를 회피.

**향후 개선 방향**
1. OM 이미지 segmentation으로 pore size, pore count, pore aspect ratio 등 도메인 특징을 정량화.
2. 미세조직 정보(α/β 비율, 결정립 크기) 추가.
3. 데이터 수 확보 후 CNN feature extractor (예: ImageNet pre-trained ResNet 임베딩) 적용.
4. Stacking / 다중 fidelity 모델 — 공정 파라미터(저해상)와 이미지(고해상)를 결합.